# 03 — Pick Best Per Class

Load the best model from each class from MLflow, generate
out-of-fold predictions for stacking, save to disk.


In [ ]:
input_dir = "data/processed"
output_dir = "data/processed"
cv_folds = 5
random_state = 42
experiments = {
    "linear": "linear_models",
    "gbdt": "gbdt_models",
    "nn": "nn_models",
}

In [ ]:
import numpy as np
import pandas as pd
import mlflow
from mlflow.tracking.client import MlflowClient
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
import joblib

In [ ]:
# ── Load data ──
X_train = pd.read_parquet(f"{input_dir}/X_train.parquet")
y_train = pd.read_parquet(f"{input_dir}/y_train.parquet")["y"]
X_test = pd.read_parquet(f"{input_dir}/X_test.parquet")
y_test = pd.read_parquet(f"{input_dir}/y_test.parquet")["y"]
X_train_scaled = StandardScaler().fit_transform(X_train)
X_test_scaled = StandardScaler().fit_transform(X_test)
print(f"Train: {len(X_train)}, Test: {len(X_test)}")

In [ ]:
client = MlflowClient()
cv = StratifiedKFold(cv_folds, shuffle=True, random_state=random_state)
oof_preds = np.zeros((len(X_train), len(experiments)))

for i, (name, experiment_name) in enumerate(experiments.items()):
    experiment = client.get_experiment_by_name(experiment_name)
    runs = client.search_runs(experiment.experiment_id, order_by=["metrics.best_cv_roc_auc DESC"])
    best_run = runs[0]
    model_uri = f"runs:/{best_run.info.run_id}/model"
    model = mlflow.sklearn.load_model(model_uri)
    print(f"Best {name}: {best_run.data.metrics['best_cv_roc_auc']:.4f} ROC-AUC")

    # Generate OOF predictions
    for train_idx, val_idx in cv.split(X_train, y_train):
        model.fit(X_train_scaled[train_idx], y_train[train_idx])
        oof_preds[val_idx, i] = model.predict_proba(X_train_scaled[val_idx])[:, 1]

    # Save test predictions too
    model.fit(X_train_scaled, y_train)
    test_preds = model.predict_proba(X_test_scaled)[:, 1]

pd.DataFrame(oof_preds, columns=list(experiments.keys())).to_parquet(
    f"{output_dir}/oof_preds.parquet"
)
pd.Series(y_train, name="match_won").to_frame().to_parquet(f"{output_dir}/y_train_full.parquet")
print(f"OOF predictions saved: {oof_preds.shape}")

In [ ]:
# ── Compare best-per-class ROC-AUC ──
from sklearn.metrics import roc_auc_score

for i, name in enumerate(experiments):
    # Approximate per-class CV score from OOF predictions
    score = roc_auc_score(y_train, oof_preds[:, i])
    print(f"  {name:8s} OOF ROC-AUC: {score:.4f}")